# CTMatch Evaluation Baseline

Establishes baseline metrics on TREC 2021, TREC 2022, and KZ datasets using the existing pipeline.
This is the regression gate — all subsequent changes must match or beat these numbers.

**Filters tested (set in Config cell):**
- `sim` — embedding + category similarity (MiniLM-L6-v2 + bart-large-mnli)
- `svm` — linear SVM on embeddings
- `demographic` — hard-exclude docs where patient age is outside the trial's stated range
- `classifier` — fine-tuned BioLinkBERT-large cross-encoder (semaj83/ctmatch-clf-biolinkbert-large)

**Metrics:** NDCG@10, MRR, F1, FPR

**Regression gate:** `sim+svm+clf` NDCG@10 ≥ 0.7528 (BioLinkBERT-large, TREC21+KZ, 134 topics)

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm evaluate lxml
!pip install -q sympy==1.13.1  # pin for torch compatibility

In [ ]:
import os
os.environ["HF_TOKEN"] = ""  # paste your WRITE token here (huggingface.co/settings/tokens)

In [ ]:
# ======== CONFIG ========
# Toggle datasets on/off
USE_TREC_2021 = True
USE_TREC_2022 = True
USE_KZ        = True

# Filter configurations to evaluate
# Add/remove configs by commenting lines in/out.
# The demographic filter is a hard age-exclusion step; it adds no latency on topics
# where patient age is not extractable, and logs how many docs it removes per topic.
FILTER_CONFIGS = {
    'sim+svm+clf':        ['sim', 'svm', 'classifier'],           # regression gate baseline
    'sim+svm+demo+clf':   ['sim', 'svm', 'demographic', 'classifier'],  # with age pre-filter
    # 'svm+clf':          ['svm', 'classifier'],                  # ablation: no sim
    # 'sim+svm+demo+clf+gen': ['sim', 'svm', 'demographic', 'classifier', 'gen'],
}

RUN_DETAILED_EVAL = True   # save per-example JSONL for error analysis (last config only)

MAX_TOPICS = None   # None = all topics across active datasets

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/ct_data23/evaluation'

# Option 2: Local
# DATA_ROOT = os.environ.get('CTMATCH_DATA_ROOT', '../data')

TREC21_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_topics.xml')
TREC21_REL_PATH   = os.path.join(DATA_ROOT, 'trec_data/trec_21_judgments.txt')
TREC22_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_data/topics2022.xml')
TREC22_REL_PATH   = os.path.join(DATA_ROOT, 'trec_data/qrels2022.txt')
KZ_TOPIC_PATH     = os.path.join(DATA_ROOT, 'kz_data/topics-2014_2015-description.topics')
KZ_REL_PATH       = os.path.join(DATA_ROOT, 'kz_data/qrels-clinical_trials.txt')

# Build active paths from config
trec_topic_paths, rel_paths = [], []
if USE_TREC_2021: trec_topic_paths.append(TREC21_TOPIC_PATH); rel_paths.append(TREC21_REL_PATH)
if USE_TREC_2022: trec_topic_paths.append(TREC22_TOPIC_PATH); rel_paths.append(TREC22_REL_PATH)
if USE_KZ:        rel_paths.append(KZ_REL_PATH)
kz_topic_path = KZ_TOPIC_PATH if USE_KZ else None

print('Active datasets:')
for p in trec_topic_paths + ([kz_topic_path] if kz_topic_path else []) + rel_paths:
    print(f"  {'✓' if os.path.exists(p) else '✗'} {p}")

## Run evaluations

In [ ]:
from ctmatch.evaluation.evaluator import EvaluatorConfig, Evaluator
from ctmatch.matching.pipeline import CTMatch
from ctmatch.config import PipeConfig

all_results = {}
configs = list(FILTER_CONFIGS.items())
for i, (name, filters) in enumerate(configs):
    print(f'Running {name}...')
    ctm = CTMatch(PipeConfig(ir_setup=True, filters=filters))
    cfg = EvaluatorConfig(
        rel_paths=rel_paths,
        trec_topic_path=trec_topic_paths or None,
        kz_topic_path=kz_topic_path,
        max_topics=MAX_TOPICS,
        filters=filters,
        ctm=ctm,
    )
    ev = Evaluator(cfg)
    # save per-example JSONL on the last config (for error analysis)
    if RUN_DETAILED_EVAL and i == len(configs) - 1:
        res = ev.evaluate_detailed(output_path='eval_predictions.jsonl')
        print(f"  saved {res['total_examples']} predictions ({res['total_errors']} errors) → eval_predictions.jsonl")
    else:
        res = ev.evaluate()
    all_results[name] = res
    for k, v in res.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
    print()

## Regression gate check

In [ ]:
GATE_CONFIG = 'sim+svm+clf'   # baseline to measure against
GATE_NDCG   = 0.7528          # BioLinkBERT-large, TREC21+KZ, 134 topics

if GATE_CONFIG in all_results:
    actual = all_results[GATE_CONFIG].get('ndcg@10', float('nan'))
    status = '✓ PASS' if actual >= GATE_NDCG else '✗ FAIL'
    print(f"{status}  {GATE_CONFIG}  NDCG@10={actual:.4f}  (gate={GATE_NDCG})")
else:
    print(f"Gate config '{GATE_CONFIG}' not in this run — add it to FILTER_CONFIGS to check.")

# Show delta for each non-gate config vs the gate
gate_ndcg = all_results.get(GATE_CONFIG, {}).get('ndcg@10')
if gate_ndcg:
    for name, res in all_results.items():
        if name == GATE_CONFIG:
            continue
        delta = res.get('ndcg@10', float('nan')) - gate_ndcg
        sign = '+' if delta >= 0 else ''
        print(f"  {name}  NDCG@10={res.get('ndcg@10', float('nan')):.4f}  delta={sign}{delta:.4f}")

## Demographic filter — per-topic exclusion counts

Shows which topics the age pre-filter fired on and how many docs it removed.
Only meaningful when `'demographic'` is in the filter list.

In [ ]:
from ctmatch.matching.demographic import extract_patient_age, trial_age_range, trial_excludes_age
import json

# Read eval_predictions.jsonl (produced by the last config)
records = []
with open('eval_predictions.jsonl') as f:
    for line in f:
        records.append(json.loads(line))

# Collect unique topics
topics = {}
for r in records:
    if r['topic_id'] not in topics:
        topics[r['topic_id']] = r['topic_text']

# Simulate filter over all records
topic_ages = {tid: extract_patient_age(text) for tid, text in topics.items()}

fired_topics = []
for tid, text in topics.items():
    pa = topic_ages[tid]
    if pa is None:
        continue
    topic_records = [r for r in records if r['topic_id'] == tid]
    excluded = [r for r in topic_records if trial_excludes_age(r['doc_text'], pa)]
    if excluded:
        fired_topics.append(dict(
            topic_id=tid,
            patient_age=f'{pa:.2f}y',
            n_excluded=len(excluded),
            n_fp_removed=sum(1 for r in excluded if r['actual'] == 0 and r['predicted'] == 2),
            n_partial_removed=sum(1 for r in excluded if r['actual'] == 1),
            n_relevant_removed=sum(1 for r in excluded if r['actual'] == 2),
        ))

import pandas as pd
df_demo = pd.DataFrame(fired_topics).set_index('topic_id')
print(f"Demographic filter fired on {len(fired_topics)} topics")
print(f"Total FP docs removed:      {df_demo['n_fp_removed'].sum()}")
print(f"Total partial docs removed: {df_demo['n_partial_removed'].sum()}")
print(f"Total RELEVANT docs removed (MUST BE 0): {df_demo['n_relevant_removed'].sum()}")
print()
df_demo

In [ ]:
import json
with open('eval_predictions.jsonl') as f:
    lines = f.readlines()
print(f'{len(lines)} total predictions')
errors = [json.loads(l) for l in lines if json.loads(l)['is_error']]
print(f'{len(errors)} errors')
print('\nFirst 3 errors:')
for ex in errors[:3]:
    print(f"  topic={ex['topic_id']} doc={ex['doc_id']} pred={ex['predicted_label']} actual={ex['actual_label']}")
    print(f"    topic: {ex['topic_text'][:100]}...")
    print(f"    doc:   {ex['doc_text'][:100]}...")
    print()

In [ ]:
# Download predictions from Colab
# from google.colab import files
# files.download('eval_predictions.jsonl')

## Summary table

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results).T[['ndcg@10', 'mean_mrr', 'mean_f1', 'mean_fpr']]
df.columns = ['NDCG@10 ↑', 'MRR ↑', 'F1 ↑', 'FPR ↓']
df.round(4)

## Results history

| Config | NDCG@10 | MRR | Notes |
|---|---|---|---|
| Original pipeline (OpenAI gen, 30 topics) | — | 0.218 | text-davinci-003 |
| sim+svm+SciBERT (TREC21+KZ, 134 topics) | 0.6525 | 0.305 | biased classifier |
| sim+svm+BioLinkBERT (TREC21+KZ, 134 topics) | **0.7528** | 0.8253 | regression gate |
| sim+svm+demo+BioLinkBERT | _TBD_ | _TBD_ | age pre-filter added |

Expected impact of demographic filter: NDCG neutral-to-small gain, FPR ↓ ~5 confirmed removals
from the 1340-record eval sweep (topics 20155, 201516, 201422, 39, 5).